# YumiCare Ultrasound Model Comparison in Colab

This notebook benchmarks multiple object-detection models on the same ultrasound validation split and exports the results to Excel.

Models included:
- YOLOv8
- YOLOv10
- EfficientDet
- SSD MobileNet
- Faster R-CNN
- Mask R-CNN

Notes:
- For a fair comparison, use fine-tuned weights for every model on the same dataset split.
- The current YumiCare dataset is YOLO bbox format, so box metrics are fully supported.
- Mask R-CNN is included for detection comparison; segmentation metrics require mask annotations.

## 1. Install Required Libraries

Install the libraries needed for detection inference, metric calculation, plotting, and Excel export in Google Colab.

In [ ]:
# Colab installs
# If a package is already available, pip will keep moving.
!pip -q install ultralytics torchmetrics pandas openpyxl seaborn matplotlib opencv-python pillow effdet timm

# TorchVision comes with Colab, but we keep the import path available for the notebook.
# If you run this outside Colab, make sure PyTorch and TorchVision are installed with GPU support when needed.

## 2. Import Libraries and Set Up Colab

Configure the runtime, select the device, and define the default project paths used by the notebook.

In [ ]:
from __future__ import annotations

import json
import math
import os
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional, Sequence, Tuple

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms
from torchvision.models.detection import (
    MaskRCNN_ResNet50_FPN_V2_Weights,
    SSD300_VGG16_Weights,
    FasterRCNN_ResNet50_FPN_V2_Weights,
    maskrcnn_resnet50_fpn_v2,
    ssd300_vgg16,
    fasterrcnn_resnet50_fpn_v2,
)
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from ultralytics import YOLO

try:
    from google.colab import drive, files
except Exception:
    drive = None
    files = None

sns.set_theme(style="whitegrid")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# Update these paths if your dataset or checkpoints live somewhere else in Colab.
COLAB_ROOT = Path("/content")
DATASET_ROOT = Path(os.environ.get("YUMICARE_DATASET_ROOT", str(COLAB_ROOT / "yumi_dataset")))
VAL_IMAGES_DIR = DATASET_ROOT / "images" / "val"
VAL_LABELS_DIR = DATASET_ROOT / "labels" / "val"
EXPERIMENT_DIR = COLAB_ROOT / "yumi_model_comparison"
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ["Brain", "CSP", "LV"]
NUM_CLASSES = len(CLASS_NAMES)
IOU_THRESHOLDS = [0.5, 0.75]
DEFAULT_INPUT_SIZE = 640

print("Dataset root:", DATASET_ROOT)
print("Validation images:", VAL_IMAGES_DIR)
print("Validation labels:", VAL_LABELS_DIR)
print("Experiment dir:", EXPERIMENT_DIR)

## 3. Load Dataset and Annotations

The current dataset is in YOLO format, so we convert the labels into absolute bounding boxes and reuse the same validation split for every model.

In [ ]:
@dataclass
class DetectionSample:
    image_path: Path
    boxes: torch.Tensor
    labels: torch.Tensor


def mount_drive_if_needed() -> None:
    if drive is not None:
        try:
            drive.mount("/content/drive")
        except Exception as exc:
            print("Drive mount skipped:", exc)


def yolo_label_file_to_target(label_path: Path, image_size: Tuple[int, int]) -> Tuple[torch.Tensor, torch.Tensor]:
    width, height = image_size
    if not label_path.exists():
        return torch.zeros((0, 4), dtype=torch.float32), torch.zeros((0,), dtype=torch.int64)

    boxes: List[List[float]] = []
    labels: List[int] = []
    with label_path.open("r", encoding="utf-8") as handle:
        for line in handle:
            stripped = line.strip()
            if not stripped:
                continue
            class_id, x_center, y_center, box_width, box_height = map(float, stripped.split())
            x1 = (x_center - box_width / 2.0) * width
            y1 = (y_center - box_height / 2.0) * height
            x2 = (x_center + box_width / 2.0) * width
            y2 = (y_center + box_height / 2.0) * height
            boxes.append([x1, y1, x2, y2])
            labels.append(int(class_id))

    if not boxes:
        return torch.zeros((0, 4), dtype=torch.float32), torch.zeros((0,), dtype=torch.int64)

    return torch.tensor(boxes, dtype=torch.float32), torch.tensor(labels, dtype=torch.int64)


def load_yolo_split(images_dir: Path, labels_dir: Path) -> List[DetectionSample]:
    samples: List[DetectionSample] = []
    if not images_dir.exists():
        raise FileNotFoundError(f"Missing images directory: {images_dir}")

    image_paths = sorted([*images_dir.glob("*.jpg"), *images_dir.glob("*.jpeg"), *images_dir.glob("*.png")])
    for image_path in image_paths:
        with Image.open(image_path) as image:
            rgb_image = image.convert("RGB")
            width, height = rgb_image.size
        label_path = labels_dir / f"{image_path.stem}.txt"
        boxes, labels = yolo_label_file_to_target(label_path, (width, height))
        samples.append(DetectionSample(image_path=image_path, boxes=boxes, labels=labels))

    print(f"Loaded {len(samples)} validation images from {images_dir}")
    return samples


mount_drive_if_needed()
val_samples = load_yolo_split(VAL_IMAGES_DIR, VAL_LABELS_DIR)
print("Example sample count:", len(val_samples))

## 4. Configure Model Zoo

Define one loader per model family. Replace the checkpoint placeholders with your fine-tuned weights for a fair comparison.

In [ ]:
MODEL_CONFIGS: Dict[str, Dict[str, Any]] = {
    "YOLOv8": {
        "kind": "ultralytics",
        "weights": "/content/weights/yolov8.pt",
        "input_size": DEFAULT_INPUT_SIZE,
    },
    "YOLOv10": {
        "kind": "ultralytics",
        "weights": "/content/weights/yolov10.pt",
        "input_size": DEFAULT_INPUT_SIZE,
    },
    "EfficientDet": {
        "kind": "effdet",
        "model_name": "tf_efficientdet_d0",
        "checkpoint": "/content/weights/efficientdet.pth",
        "input_size": 512,
    },
    "SSD MobileNet": {
        "kind": "torchvision_ssd",
        "checkpoint": "/content/weights/ssd_mobilenet.pth",
        "input_size": 300,
    },
    "Faster R-CNN": {
        "kind": "torchvision_fasterrcnn",
        "checkpoint": "/content/weights/fasterrcnn.pth",
        "input_size": 640,
    },
    "Mask R-CNN": {
        "kind": "torchvision_maskrcnn",
        "checkpoint": "/content/weights/maskrcnn.pth",
        "input_size": 640,
    },
}


def build_ultralytics_model(weights_path: str) -> YOLO:
    if not Path(weights_path).exists():
        raise FileNotFoundError(
            f"Missing weights: {weights_path}. Update MODEL_CONFIGS with your fine-tuned checkpoint path."
        )
    return YOLO(weights_path)


def build_effdet_model(model_name: str, checkpoint: str) -> torch.nn.Module:
    from effdet import create_model

    model = create_model(model_name, bench_task="predict", pretrained=False, num_classes=NUM_CLASSES)
    if Path(checkpoint).exists():
        state_dict = torch.load(checkpoint, map_location="cpu")
        model.load_state_dict(state_dict, strict=False)
    model.eval().to(DEVICE)
    return model


def build_torchvision_model(kind: str, checkpoint: str) -> torch.nn.Module:
    if kind == "torchvision_ssd":
        model = ssd300_vgg16(weights=None, weights_backbone=None, num_classes=NUM_CLASSES + 1)
    elif kind == "torchvision_fasterrcnn":
        model = fasterrcnn_resnet50_fpn_v2(weights=None, weights_backbone=None, num_classes=NUM_CLASSES + 1)
    elif kind == "torchvision_maskrcnn":
        model = maskrcnn_resnet50_fpn_v2(weights=None, weights_backbone=None, num_classes=NUM_CLASSES + 1)
    else:
        raise ValueError(f"Unsupported torchvision model kind: {kind}")

    if Path(checkpoint).exists():
        checkpoint_state = torch.load(checkpoint, map_location="cpu")
        model.load_state_dict(checkpoint_state, strict=False)
    else:
        print(f"No fine-tuned checkpoint found for {kind}; model is initialized without pretrained weights.")

    model.eval().to(DEVICE)
    return model


MODEL_OBJECTS: Dict[str, Any] = {}
for model_name, config in MODEL_CONFIGS.items():
    try:
        if config["kind"] == "ultralytics":
            MODEL_OBJECTS[model_name] = build_ultralytics_model(config["weights"])
        elif config["kind"] == "effdet":
            MODEL_OBJECTS[model_name] = build_effdet_model(config["model_name"], config["checkpoint"])
        else:
            MODEL_OBJECTS[model_name] = build_torchvision_model(config["kind"], config["checkpoint"])
        print(f"Loaded {model_name}")
    except Exception as exc:
        MODEL_OBJECTS[model_name] = exc
        print(f"Could not load {model_name}: {exc}")

## 5. Run Inference for All Models

This cell defines a single inference interface for every model family and stores boxes, classes, scores, and masks when available.

In [ ]:
def read_image_as_tensor(image_path: Path, input_size: int) -> Tuple[torch.Tensor, Tuple[int, int]]:
    with Image.open(image_path) as image:
        rgb_image = image.convert("RGB")
        width, height = rgb_image.size
        resized = rgb_image.resize((input_size, input_size))
        array = np.asarray(resized).astype(np.float32) / 255.0
        tensor = torch.from_numpy(array).permute(2, 0, 1)
    return tensor, (width, height)


TORCHVISION_TRANSFORM = transforms.Compose([transforms.ToTensor()])


def ultralytics_predict(model: YOLO, image_path: Path) -> Dict[str, torch.Tensor]:
    result = model.predict(source=str(image_path), verbose=False)[0]
    boxes = result.boxes.xyxy.detach().cpu() if result.boxes is not None else torch.zeros((0, 4))
    scores = result.boxes.conf.detach().cpu() if result.boxes is not None else torch.zeros((0,))
    labels = result.boxes.cls.detach().cpu().to(torch.int64) if result.boxes is not None else torch.zeros((0,), dtype=torch.int64)
    masks = None
    if getattr(result, "masks", None) is not None:
        masks = result.masks.data.detach().cpu()
    return {"boxes": boxes, "scores": scores, "labels": labels, "masks": masks}


def torchvision_predict(model: torch.nn.Module, image_path: Path, input_size: int) -> Dict[str, torch.Tensor]:
    image_tensor, _ = read_image_as_tensor(image_path, input_size)
    image_tensor = image_tensor.to(DEVICE)
    with torch.no_grad():
        outputs = model([image_tensor])[0]
    boxes = outputs.get("boxes", torch.zeros((0, 4), device=DEVICE)).detach().cpu()
    scores = outputs.get("scores", torch.zeros((0,), device=DEVICE)).detach().cpu()
    labels = outputs.get("labels", torch.zeros((0,), device=DEVICE)).detach().cpu().to(torch.int64)
    masks = outputs.get("masks")
    if masks is not None:
        masks = masks.detach().cpu()
    return {"boxes": boxes, "scores": scores, "labels": labels, "masks": masks}


def effdet_predict(model: torch.nn.Module, image_path: Path, input_size: int) -> Dict[str, torch.Tensor]:
    image_tensor, _ = read_image_as_tensor(image_path, input_size)
    image_tensor = image_tensor.to(DEVICE)
    with torch.no_grad():
        outputs = model([image_tensor]) if callable(getattr(model, "__call__", None)) else model(image_tensor)
    if isinstance(outputs, dict):
        boxes = outputs.get("boxes", torch.zeros((0, 4), device=DEVICE)).detach().cpu()
        scores = outputs.get("scores", torch.zeros((0,), device=DEVICE)).detach().cpu()
        labels = outputs.get("labels", torch.zeros((0,), device=DEVICE)).detach().cpu().to(torch.int64)
    else:
        boxes = torch.zeros((0, 4))
        scores = torch.zeros((0,))
        labels = torch.zeros((0,), dtype=torch.int64)
    return {"boxes": boxes, "scores": scores, "labels": labels, "masks": None}


def run_single_model_inference(model_name: str, model_object: Any, sample: DetectionSample) -> Dict[str, torch.Tensor]:
    config = MODEL_CONFIGS[model_name]
    if isinstance(model_object, Exception):
        raise model_object
    if config["kind"] == "ultralytics":
        return ultralytics_predict(model_object, sample.image_path)
    if config["kind"] == "effdet":
        return effdet_predict(model_object, sample.image_path, config["input_size"])
    return torchvision_predict(model_object, sample.image_path, config["input_size"])


# This dictionary will hold per-model per-image predictions.
PREDICTIONS: Dict[str, List[Dict[str, torch.Tensor]]] = {name: [] for name in MODEL_OBJECTS}

for model_name, model_object in MODEL_OBJECTS.items():
    if isinstance(model_object, Exception):
        print(f"Skipping {model_name} because it failed to load.")
        continue
    for sample in val_samples:
        try:
            PREDICTIONS[model_name].append(run_single_model_inference(model_name, model_object, sample))
        except Exception as exc:
            print(f"Inference failed for {model_name} on {sample.image_path.name}: {exc}")
            PREDICTIONS[model_name].append({"boxes": torch.zeros((0, 4)), "scores": torch.zeros((0,)), "labels": torch.zeros((0,), dtype=torch.int64), "masks": None})
    print(f"Completed inference for {model_name}")

## 6. Compute Evaluation Metrics

We compute detection metrics on the validation split using the same matching rules for every model. Box metrics are used for Mask R-CNN because the current dataset does not include mask annotations.

In [ ]:
def box_iou(box_a: torch.Tensor, box_b: torch.Tensor) -> torch.Tensor:
    if box_a.numel() == 0 or box_b.numel() == 0:
        return torch.zeros((box_a.shape[0], box_b.shape[0]), dtype=torch.float32)
    area_a = (box_a[:, 2] - box_a[:, 0]).clamp(min=0) * (box_a[:, 3] - box_a[:, 1]).clamp(min=0)
    area_b = (box_b[:, 2] - box_b[:, 0]).clamp(min=0) * (box_b[:, 3] - box_b[:, 1]).clamp(min=0)
    lt = torch.maximum(box_a[:, None, :2], box_b[:, :2])
    rb = torch.minimum(box_a[:, None, 2:], box_b[:, 2:])
    wh = (rb - lt).clamp(min=0)
    inter = wh[:, :, 0] * wh[:, :, 1]
    union = area_a[:, None] + area_b - inter + 1e-9
    return inter / union


def greedy_match_counts(pred_boxes: torch.Tensor, pred_scores: torch.Tensor, gt_boxes: torch.Tensor, iou_threshold: float = 0.5) -> Tuple[int, int, int]:
    if pred_boxes.numel() == 0:
        return 0, 0, int(gt_boxes.shape[0])
    if gt_boxes.numel() == 0:
        return 0, int(pred_boxes.shape[0]), 0

    order = torch.argsort(pred_scores, descending=True)
    pred_boxes = pred_boxes[order]
    ious = box_iou(pred_boxes, gt_boxes)
    matched_gt: set[int] = set()
    true_positive = 0
    false_positive = 0
    for pred_idx in range(pred_boxes.shape[0]):
        best_iou = 0.0
        best_gt = -1
        for gt_idx in range(gt_boxes.shape[0]):
            if gt_idx in matched_gt:
                continue
            current_iou = float(ious[pred_idx, gt_idx])
            if current_iou > best_iou:
                best_iou = current_iou
                best_gt = gt_idx
        if best_iou >= iou_threshold and best_gt >= 0:
            true_positive += 1
            matched_gt.add(best_gt)
        else:
            false_positive += 1
    false_negative = gt_boxes.shape[0] - len(matched_gt)
    return true_positive, false_positive, false_negative


@torch.no_grad()
def compute_map_metrics(predictions: List[Dict[str, torch.Tensor]], samples: List[DetectionSample]) -> Dict[str, float]:
    metric = MeanAveragePrecision(box_format="xyxy", iou_type="bbox", class_metrics=True)
    for prediction, sample in zip(predictions, samples):
        metric.update(
            [
                {
                    "boxes": prediction["boxes"],
                    "scores": prediction["scores"],
                    "labels": prediction["labels"],
                }
            ],
            [
                {
                    "boxes": sample.boxes,
                    "labels": sample.labels,
                }
            ],
        )
    result = metric.compute()
    return {
        "mAP@50": float(result["map_50"].item()) if result["map_50"] is not None else float("nan"),
        "mAP@50:95": float(result["map"].item()) if result["map"] is not None else float("nan"),
        "per_class_map50": result.get("map_per_class", torch.tensor([])).detach().cpu().tolist() if result.get("map_per_class") is not None else [],
    }


def compute_precision_recall_f1(predictions: List[Dict[str, torch.Tensor]], samples: List[DetectionSample], iou_threshold: float = 0.5) -> Dict[str, float]:
    true_positive = 0
    false_positive = 0
    false_negative = 0
    for prediction, sample in zip(predictions, samples):
        tp, fp, fn = greedy_match_counts(prediction["boxes"], prediction["scores"], sample.boxes, iou_threshold=iou_threshold)
        true_positive += tp
        false_positive += fp
        false_negative += fn
    precision = true_positive / max(true_positive + false_positive, 1)
    recall = true_positive / max(true_positive + false_negative, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-9)
    return {"precision": precision, "recall": recall, "F1": f1}


def compute_class_counts(samples: List[DetectionSample]) -> Dict[str, int]:
    counts = {name: 0 for name in CLASS_NAMES}
    for sample in samples:
        for label in sample.labels.tolist():
            if 0 <= label < len(CLASS_NAMES):
                counts[CLASS_NAMES[label]] += 1
    return counts

## 7. Measure Speed and Resource Usage

Measure average latency, FPS, model size, and GPU memory usage where supported.

In [ ]:
def get_model_size_mb(model_object: Any) -> float:
    if isinstance(model_object, YOLO):
        weights_path = getattr(model_object, "ckpt_path", None)
        if weights_path:
            path = Path(str(weights_path))
            if path.exists():
                return path.stat().st_size / (1024 * 1024)
        return float("nan")
    if hasattr(model_object, "state_dict"):
        temp_path = EXPERIMENT_DIR / "temp_model_size.pth"
        torch.save(model_object.state_dict(), temp_path)
        size_mb = temp_path.stat().st_size / (1024 * 1024)
        temp_path.unlink(missing_ok=True)
        return size_mb
    return float("nan")


def benchmark_predictions(model_name: str, model_object: Any, samples: List[DetectionSample]) -> Dict[str, float]:
    if isinstance(model_object, Exception):
        return {
            "latency_ms": float("nan"),
            "fps": float("nan"),
            "gpu_mem_mb": float("nan"),
            "model_size_mb": float("nan"),
        }

    input_size = MODEL_CONFIGS[model_name]["input_size"]
    warmup_count = min(2, len(samples))
    benchmark_samples = samples[: min(10, len(samples))] if len(samples) > 0 else []
    for sample in samples[:warmup_count]:
        _ = run_single_model_inference(model_name, model_object, sample)

    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()

    timings: List[float] = []
    for sample in benchmark_samples:
        start = time.perf_counter()
        _ = run_single_model_inference(model_name, model_object, sample)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        timings.append((time.perf_counter() - start) * 1000.0)

    latency_ms = float(np.mean(timings)) if timings else float("nan")
    fps = 1000.0 / latency_ms if latency_ms and not math.isnan(latency_ms) and latency_ms > 0 else float("nan")
    gpu_mem_mb = float(torch.cuda.max_memory_allocated() / (1024 * 1024)) if DEVICE.type == "cuda" else 0.0
    model_size_mb = get_model_size_mb(model_object)
    return {
        "latency_ms": latency_ms,
        "fps": fps,
        "gpu_mem_mb": gpu_mem_mb,
        "model_size_mb": model_size_mb,
    }

## 8. Collect Results into a Pandas DataFrame

Run the evaluation loop for each model and collect the metrics into one summary table.

In [ ]:
RESULT_ROWS: List[Dict[str, Any]] = []
PER_CLASS_ROWS: List[Dict[str, Any]] = []

for model_name, model_object in MODEL_OBJECTS.items():
    if isinstance(model_object, Exception):
        RESULT_ROWS.append(
            {
                "model": model_name,
                "status": f"load_failed: {model_object}",
                "precision": float("nan"),
                "recall": float("nan"),
                "F1": float("nan"),
                "mAP@50": float("nan"),
                "mAP@50:95": float("nan"),
                "latency_ms": float("nan"),
                "fps": float("nan"),
                "gpu_mem_mb": float("nan"),
                "model_size_mb": float("nan"),
            }
        )
        continue

    detection_metrics = compute_precision_recall_f1(PREDICTIONS[model_name], val_samples)
    map_metrics = compute_map_metrics(PREDICTIONS[model_name], val_samples)
    speed_metrics = benchmark_predictions(model_name, model_object, val_samples)

    row = {
        "model": model_name,
        "status": "ok",
        **detection_metrics,
        "mAP@50": map_metrics["mAP@50"],
        "mAP@50:95": map_metrics["mAP@50:95"],
        **speed_metrics,
    }
    RESULT_ROWS.append(row)

    per_class_map = map_metrics.get("per_class_map50", [])
    class_supports = compute_class_counts(val_samples)
    for class_index, class_name in enumerate(CLASS_NAMES):
        class_map50 = float(per_class_map[class_index]) if class_index < len(per_class_map) else float("nan")
        PER_CLASS_ROWS.append(
            {
                "model": model_name,
                "class_name": class_name,
                "support": class_supports.get(class_name, 0),
                "map50": class_map50,
            }
        )

results_df = pd.DataFrame(RESULT_ROWS)
results_df = results_df.sort_values(by=["mAP@50", "F1"], ascending=False, na_position="last").reset_index(drop=True)
results_df

## 9. Export Metrics to Excel

Write the comparison table to an Excel workbook with a summary sheet and supporting detail sheets.

In [ ]:
excel_path = EXPERIMENT_DIR / "yumi_ultrasound_model_comparison.xlsx"
csv_path = EXPERIMENT_DIR / "yumi_ultrasound_model_comparison.csv"
per_class_df = pd.DataFrame(PER_CLASS_ROWS)

results_df.to_csv(csv_path, index=False)
with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    results_df.to_excel(writer, sheet_name="summary", index=False)
    per_class_df.to_excel(writer, sheet_name="per_class", index=False)
    pd.DataFrame(
        {
            "key": [
                "dataset_root",
                "validation_images",
                "validation_labels",
                "class_names",
                "device",
            ],
            "value": [
                str(DATASET_ROOT),
                str(VAL_IMAGES_DIR),
                str(VAL_LABELS_DIR),
                ", ".join(CLASS_NAMES),
                str(DEVICE),
            ],
        }
    ).to_excel(writer, sheet_name="settings", index=False)

print("Saved CSV:", csv_path)
print("Saved Excel:", excel_path)

## 10. Save and Download the Excel File in Colab

Persist the Excel file in the Colab working directory and create a direct download link when running inside Colab.

In [ ]:
if files is not None and excel_path.exists():
    files.download(str(excel_path))
else:
    print("Download helper is only available inside Google Colab.")

## 11. Compare Model Results with Tables and Plots

Create a quick visual comparison for precision, recall, mAP, latency, FPS, and model size.

In [ ]:
plot_columns = [
    "precision",
    "recall",
    "F1",
    "mAP@50",
    "mAP@50:95",
    "latency_ms",
    "fps",
    "model_size_mb",
]

plot_df = results_df.copy()
fig, axes = plt.subplots(2, 4, figsize=(22, 10))
axes = axes.flatten()
for axis, column in zip(axes, plot_columns):
    sns.barplot(data=plot_df, x="model", y=column, ax=axis, palette="viridis")
    axis.set_title(column)
    axis.tick_params(axis="x", rotation=30)
    if column in {"latency_ms", "model_size_mb"}:
        axis.set_ylabel(column)
plt.tight_layout()
plt.show()

heatmap_df = plot_df.set_index("model")[["precision", "recall", "F1", "mAP@50", "mAP@50:95", "fps"]]
plt.figure(figsize=(12, 6))
sns.heatmap(heatmap_df, annot=True, fmt=".3f", cmap="mako")
plt.title("Model Comparison Heatmap")
plt.tight_layout()
plt.show()

results_df